# NVFP4 GEMM on Colab G4: 양자화·정확성·BF16 성능 비교

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jake-Song/tensor2silicon/blob/main/notebooks/nvfp4_gemm_g4.ipynb)

Colab에서 **런타임 → 런타임 유형 변경 → G4**를 선택하고 위에서부터 실행한다. [BF16 G4 노트북](triton_gemm_bf16_g4.ipynb)에 이어, 이번에는 **NVFP4 입력을 받는 네이티브 GEMM**을 같은 장치의 BF16 GEMM과 비교한다. 출력은 모두 BF16이다. 사용자 Triton GEMM의 v1~v6 비교는 앞 노트북에 있고, 이 노트북은 PyTorch의 cuBLAS 경로로 FP4 형식과 양자화 비용을 먼저 확인한다.

- 대상: NVIDIA RTX PRO 6000 Blackwell Server Edition, **sm_120**.
- 기준 환경: 앞 실험의 PyTorch **2.11.0+cu130**, Triton **3.6.0**. `torch.nn.functional.scaled_mm` API를 사용한다.
- 순서: FP4 표현 → 2단계 스케일 → 패킹/배치 검사 → 네이티브 GEMM 검사 → 정확성 → 128~8192 벤치마크 → JSON 저장.
- G4 실측값과 정확성 결과는 마지막 8절에 정리했다. 네이티브 GEMM이 지원되지 않으면 오류로 중단하며, BF16 복원 연산을 FP4 성능으로 표시하지 않는다.

## 0. 환경 확인

첫 셀에서 GPU와 API를 확인한다. API가 없으면 PyTorch 2.11.0+cu130 환경을 준비하고 런타임을 다시 시작해야 한다. 실행 중인 PyTorch를 자동 교체하지 않는다. 이후 3절의 작은 GEMM 호출이 실제 CUDA 라이브러리 지원 여부까지 확인한다.

In [ ]:
import json, math, re, statistics
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
import torch
import torch.nn.functional as F
import triton
import triton.testing
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

assert torch.cuda.is_available(), "Colab G4 GPU 런타임이 필요합니다."
p = torch.cuda.get_device_properties(0)
assert "RTX PRO 6000" in p.name and "Blackwell" in p.name and (p.major, p.minor) == (12, 0), p.name
assert tuple(map(int, torch.__version__.split("+")[0].split(".")[:2])) >= (2, 11)
assert all(hasattr(F, n) for n in ("scaled_mm", "ScalingType", "SwizzleType"))
assert hasattr(torch, "float4_e2m1fn_x2")
torch.set_grad_enabled(False)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
STARTED_UTC = datetime.now(timezone.utc).isoformat()
ENVIRONMENT = dict(gpu=p.name, compute_capability=f"{p.major}.{p.minor}",
                   sms=p.multi_processor_count, memory_gib=p.total_memory / 2**30,
                   torch=torch.__version__, cuda=torch.version.cuda, triton=triton.__version__)
print(json.dumps(ENVIRONMENT, indent=2))

## 1. FP4는 값 4비트 + 스케일이다

E2M1은 부호 1비트·지수 2비트·가수 1비트로, 절댓값은 **0, 0.5, 1, 1.5, 2, 3, 4, 6** 중 하나다. 두 값을 `uint8` 하나의 하위/상위 4비트에 넣는다. NVFP4는 K축의 **16개 원소마다 FP8 E4M3 스케일** 하나와 텐서 전체의 FP32 스케일 하나를 사용한다. MXFP4의 32원소/E8M0 스케일과 구분한다.

$$\hat{x}_{r,k}=q_{r,k}\,s_{r,\lfloor k/16\rfloor}\,g,\qquad q\in\mathrm{E2M1},\ s\in\mathrm{E4M3},\ g\in\mathrm{FP32}$$

이 노트북의 참조 양자화기는 $g=\max|x|/(6\times448)$, $s=\mathrm{E4M3}(\max_{16}|x|/(6g))$를 구하고, **실제로 저장된 s**로 나눈 값을 가장 가까운 E2M1 값에 반올림한다. 정확히 중간이면 짝수 코드로 반올림한다. 전체가 0인 입력은 g=1로 처리하고, 너무 작은 블록 스케일은 E4M3 최소 양수 $2^{-9}$로 제한한다. 미세한 값의 underflow는 양자화 오차에 포함한다. 입력은 유한한 BF16 값으로 한정한다.

패딩을 제외하면 원소당 $4+8/16=4.5$비트로, BF16 대비 입력 저장량은 약 **3.56배 감소**한다. 실제 메모리에는 스케일 패딩·FP32 전역 스케일·BF16 출력·양자화 임시 버퍼도 들어간다. 저장량 비율이 곧 속도 향상 비율은 아니다.

스케일 형식과 배치의 근거: [NVIDIA cuBLAS block scaling](https://docs.nvidia.com/cuda/cublas/index.html#d-block-scaling-factors-layout), [PyTorch 2.11 NVFP4 검증 코드](https://github.com/pytorch/pytorch/blob/v2.11.0/test/test_scaled_matmul_cuda.py).

In [ ]:
# 이 셀의 참조 함수들은 CPU에서도 검증할 수 있다.
@dataclass
class NVFP4:
    packed: torch.Tensor        # (rows, K/2), uint8
    scales: torch.Tensor        # (rows, K/16), E4M3, 논리 배치
    global_scale: torch.Tensor  # (1,), FP32, 복원 시 곱하는 값
    blocked_scales: torch.Tensor  # cuBLAS SWIZZLE_32_4_4 배치


def encode_e2m1(x):
    boundaries = torch.tensor([.25, .75, 1.25, 1.75, 2.5, 3.5, 5.], device=x.device)
    mag = x.float().abs().clamp(max=6).contiguous()
    lower = torch.bucketize(mag, boundaries, right=False)
    ties = mag == boundaries[lower.clamp(max=6)]
    code = lower + (ties & ((lower & 1) == 1)).to(lower.dtype)
    return code.to(torch.uint8) | (torch.signbit(x).to(torch.uint8) << 3)


def pack_e2m1(codes):
    assert codes.shape[-1] % 2 == 0
    return (codes[..., 0::2] | (codes[..., 1::2] << 4)).contiguous()


def decode_e2m1(packed):
    codes = torch.stack((packed & 15, packed >> 4), dim=-1).flatten(-2)
    values = torch.tensor([0., .5, 1., 1.5, 2., 3., 4., 6.], device=packed.device)
    return values[(codes & 7).long()] * torch.where((codes & 8) != 0, -1., 1.)


def swizzle_scales(scales):
    # 128 rows × 4 scale columns을 [32, 4, 4] 순서로 저장한다.
    # byte view를 사용해 FP8에서 지원되지 않는 연산을 피한다.
    rows, cols = scales.shape
    rp, cp = triton.cdiv(rows, 128) * 128, triton.cdiv(cols, 4) * 4
    padded = torch.zeros((rp, cp), dtype=torch.uint8, device=scales.device)
    padded[:rows, :cols] = scales.view(torch.uint8)
    blocked = padded.reshape(rp // 128, 4, 32, cp // 4, 4)
    return blocked.permute(0, 3, 2, 1, 4).contiguous().flatten().view(torch.float8_e4m3fn)


def quantize_nvfp4(x):
    assert x.ndim == 2 and x.dtype == torch.bfloat16 and x.shape[1] % 16 == 0
    # 호출 전에 유한성을 검증한다. 측정 경로에 .item()/CPU 동기화를 넣지 않는다.
    rows, k = x.shape
    blocks = x.float().reshape(rows, k // 16, 16)
    global_max = blocks.abs().amax()
    g = torch.where(global_max > 0, global_max / (6 * 448), torch.ones_like(global_max)).reshape(1)
    g = g.clamp_min(torch.finfo(torch.float32).tiny)  # 전역 스케일의 0 underflow 방지
    local = (blocks.abs().amax(dim=-1) / (6 * g)).clamp(min=2**-9, max=448)
    s = local.to(torch.float8_e4m3fn)
    normalized = blocks / (s.float() * g).unsqueeze(-1)
    q = pack_e2m1(encode_e2m1(normalized.reshape(rows, k)))
    return NVFP4(q, s, g, swizzle_scales(s))


def dequantize_nvfp4(q):
    rows, half_k = q.packed.shape
    values = decode_e2m1(q.packed).reshape(rows, -1, 16)
    return (values * q.scales.float().unsqueeze(-1) * q.global_scale).reshape(rows, half_k * 2)

## 2. 패킹과 스케일 배치를 먼저 검사한다

16개 코드 전체, 중간값 반올림, 하위 nibble 순서, 패딩된 스케일의 주소를 검사한다. 잘못된 스케일 배치는 GEMM 결과를 크게 바꾸므로, 단순 pack/unpack 왕복 외에 논리 좌표에서 계산한 주소와도 비교한다. 양자화기는 설명을 위한 eager PyTorch 구현으로, 여러 커널과 임시 버퍼를 사용한다.

In [ ]:
def test_format(device="cpu"):
    values = torch.tensor([0., .5, 1., 1.5, 2., 3., 4., 6.,
                           -0., -.5, -1., -1.5, -2., -3., -4., -6.], device=device)
    codes = encode_e2m1(values)
    assert torch.equal(codes, torch.arange(16, device=device, dtype=torch.uint8))
    packed = pack_e2m1(codes)
    assert packed[0].item() == 0x10  # 첫 원소가 하위 4비트
    torch.testing.assert_close(decode_e2m1(packed), values, rtol=0, atol=0)
    ties = torch.tensor([.25, .75, 1.25, 1.75, 2.5, 3.5, 5., 8.], device=device)
    expected = torch.tensor([0, 2, 2, 4, 4, 6, 6, 7], device=device, dtype=torch.uint8)
    assert torch.equal(encode_e2m1(ties), expected)
    assert torch.equal(encode_e2m1(-ties), expected | 8)

    rows, cols = 137, 7
    scales = ((torch.arange(rows * cols, device=device).reshape(rows, cols) % 200) / 8).to(torch.float8_e4m3fn)
    got = swizzle_scales(scales).view(torch.uint8)
    r = torch.arange(rows, device=device)[:, None]
    c = torch.arange(cols, device=device)[None, :]
    address = (((r // 128) * triton.cdiv(cols, 4) + c // 4) * 512
               + (r % 32) * 16 + ((r % 128) // 32) * 4 + c % 4)
    assert torch.equal(got[address], scales.view(torch.uint8))
    used = torch.zeros_like(got, dtype=torch.bool)
    used[address] = True
    assert not got[~used].any().item()

    for multiplier in [0., 1e-6, 1., 1e3]:
        x = (torch.linspace(-1, 1, 128 * 64, device=device).reshape(128, 64) * multiplier).bfloat16()
        q = quantize_nvfp4(x)
        restored = dequantize_nvfp4(q)
        assert torch.isfinite(restored).all().item()
        assert torch.isfinite(q.scales.float()).all().item()
        if multiplier == 0:
            assert torch.count_nonzero(restored).item() == 0
        else:
            nrmse = (restored - x.float()).norm() / x.float().norm()
            assert nrmse.item() < .2, nrmse
    return dict(device=str(device), ok=True)

format_checks = [test_format("cpu"), test_format("cuda")]
print(format_checks)

## 3. 네이티브 NVFP4 GEMM

A는 (M,K), W는 (N,K)로 저장하고 $C=AW^T$를 계산한다. 두 입력 모두 K축을 따라 2개씩 패킹하며, W는 **패킹 후 transpose view**로 GEMM에 전달한다. 스케일은 두 입력 모두 논리 (행,K/16)에서 cuBLAS 배치로 변환한다. 전역 스케일도 GEMM API에 넘기므로 출력에서 별도 보정 커널을 실행하지 않는다.

[PyTorch 2.11 `scaled_mm`](https://docs.pytorch.org/docs/2.11/generated/torch.nn.functional.scaled_mm.html)의 `BlockWise1x16 + TensorWise`, `SWIZZLE_32_4_4 + NO_SWIZZLE` 조합을 사용한다. `use_fast_accum=False`, BF16 출력으로 실행한다. 이 노트북은 설명을 단순하게 유지하기 위해 M,N은 128의 배수, K는 64의 배수만 벤치마크한다.

In [ ]:
def nvfp4_mm(aq, wq):
    m, half_k = aq.packed.shape
    n, other_half_k = wq.packed.shape
    assert half_k == other_half_k and m % 128 == n % 128 == 0 and (2 * half_k) % 64 == 0
    recipe = [F.ScalingType.BlockWise1x16, F.ScalingType.TensorWise]
    swizzle = [F.SwizzleType.SWIZZLE_32_4_4, F.SwizzleType.NO_SWIZZLE]
    return F.scaled_mm(
        aq.packed.view(torch.float4_e2m1fn_x2),
        wq.packed.view(torch.float4_e2m1fn_x2).t(),
        scale_a=[aq.blocked_scales, aq.global_scale], scale_recipe_a=recipe,
        scale_b=[wq.blocked_scales, wq.global_scale], scale_recipe_b=recipe,
        swizzle_a=swizzle, swizzle_b=swizzle,
        output_dtype=torch.bfloat16, use_fast_accum=False,
    )


def make_inputs(m, n, k, seed=0, device="cuda"):
    generator = torch.Generator(device=device).manual_seed(seed)
    a = torch.randn(m, k, generator=generator, device=device).bfloat16()
    w = (torch.randn(n, k, generator=generator, device=device) / math.sqrt(k)).bfloat16()
    return a, w


def prepare(a, w):
    assert torch.isfinite(a).all().item() and torch.isfinite(w).all().item()
    return quantize_nvfp4(a), quantize_nvfp4(w)


a, w = make_inputs(128, 128, 128)
aq, wq = prepare(a, w)
try:
    smoke = nvfp4_mm(aq, wq)
    torch.cuda.synchronize()
except (RuntimeError, NotImplementedError) as exc:
    raise RuntimeError(
        f"네이티브 NVFP4 GEMM을 실행하지 못했습니다. 환경: {ENVIRONMENT}. "
        "PyTorch 2.11.0+cu130 및 G4를 확인하세요. BF16 대체 측정은 수행하지 않습니다."
    ) from exc
assert smoke.shape == (128, 128) and smoke.dtype == torch.bfloat16
assert torch.isfinite(smoke).all().item()
print("NVFP4 native GEMM smoke test passed")

## 4. 구현 오차와 양자화 오차를 분리한다

두 가지 FP32 기준값을 사용한다.

1. **구현 검증**: 패킹된 값을 직접 복원한 $\hat A\hat W^T$와 네이티브 FP4 출력을 비교한다. 패킹·스케일·행렬 방향 오류를 잡는다. 출력 BF16 반올림을 고려해 `atol=rtol=2e-2`를 사용한다.
2. **양자화 영향**: 양자화 전 BF16 값을 FP32로 올린 $AW^T$와 비교한다. 여기는 BF16 수준의 `allclose`를 강제하지 않고 NRMSE와 SQNR을 기록한다. 작은 원소의 상대오차 폭증을 피하기 위해 전체 norm 기준도 함께 본다.

$$\mathrm{NRMSE}=\|C-C_{ref}\|_2/\|C_{ref}\|_2,\quad
\mathrm{SQNR}=20\log_{10}(\|C_{ref}\|_2/\|C-C_{ref}\|_2)$$

SQNR은 높을수록 좋다. 0 기준값이나 오차가 정확히 0인 경우에는 SQNR을 `None`으로 저장한다. 무작위 입력의 오차가 실제 모델의 품질을 보장하지는 않는다.

In [ ]:
def error_metrics(actual, reference):
    delta = actual.float() - reference.float()
    err, norm = delta.norm().item(), reference.float().norm().item()
    return dict(max_abs_error=delta.abs().max().item(),
                nrmse=err / max(norm, 1e-30),
                sqnr_db=20 * math.log10(norm / err) if err > 0 and norm > 0 else None)


def validate_case(m, n, k, kind="random"):
    a, w = make_inputs(m, n, k, seed=m + 7 * n + 13 * k)
    if kind == "zero":
        a.zero_()
    elif kind == "mixed_range":
        # 같은 텐서 안에 작은 값과 이상치를 섞어 스케일 범위를 검사한다.
        a[:, :16] *= 1e-5
        a[:, 16:32] *= 32
    aq, wq = prepare(a, w)
    actual = nvfp4_mm(aq, wq)
    reconstructed = dequantize_nvfp4(aq) @ dequantize_nvfp4(wq).t()
    original = a.float() @ w.float().t()
    assert torch.isfinite(actual).all().item()
    torch.testing.assert_close(actual.float(), reconstructed, atol=2e-2, rtol=2e-2)
    return dict(m=m, n=n, k=k, kind=kind, ok=True,
                implementation=error_metrics(actual, reconstructed),
                quantization_only=error_metrics(reconstructed, original),
                total_fp4=error_metrics(actual, original),
                bf16=error_metrics(a @ w.t(), original))


CHECK_CASES = [(128, 128, 128, "random"), (256, 512, 1024, "random"),
               (1024, 256, 512, "random"), (128, 128, 128, "zero"),
               (256, 128, 256, "mixed_range")]
accuracy_rows = [validate_case(*case) for case in CHECK_CASES]
display(pd.json_normalize(accuracy_rows))
print("Packing/scaling/native GEMM correctness checks passed")

## 5. 크기별 성능: 미리 양자화한 GEMM과 전체 경로

두 입력 모두 GPU에 있는 상태에서 측정한다. `triton.testing.do_bench`의 cache-flush median을 사용하고, 각 경로를 3회 측정해 median과 실행 간 최소/최대를 기록한다. 경로 순서는 반복마다 섞는다. 컴파일·최초 라이브러리 초기화는 측정 전에 끝낸다. 반환 출력 할당을 포함하는 API 호출 비교이며 순수 커널 시간만 분리한 값은 아니다.

| 경로 | 포함하는 작업 |
|---|---|
| BF16 GEMM | 원래 BF16 A·W의 `torch.matmul` |
| NVFP4 GEMM | 미리 양자화·패킹·스케일 배치를 마친 입력의 네이티브 GEMM |
| Quantize A+W | BF16 두 입력의 참조 양자화·패킹·스케일 배치 |
| NVFP4 end-to-end | 두 입력 양자화부터 네이티브 GEMM까지 |

기본 정사각 크기는 128~8192다. 느린 참조 양자화의 전체 경로는 기본 2048까지만 측정하며 `E2E_MAX`로 늘릴 수 있다. 이 참조 구현의 비용을 최적화된 FP4 양자화기의 한계로 해석하지 않는다. 실서비스에서 W를 미리 양자화하는 경우의 비용과도 다르다.

앞 BF16 노트북은 B를 (K,N) row-major로 저장하지만 여기서는 W를 (N,K)로 저장한 뒤 transpose view를 넘긴다. cuBLAS의 layout별 선택이 달라질 수 있으므로 **이 노트북 안에서 새로 측정한 BF16**을 기준으로 한다. `2MNK/time`은 유효 GEMM 처리량이며, 전체 경로에는 양자화 연산의 FLOP 수를 더하지 않는다.

In [ ]:
import random
SIZES = [128, 256, 512, 1024, 2048, 4096, 8192]
E2E_MAX = 2048
REPEATS, WARMUP_MS, REP_MS = 3, 100, 250
benchmark_rows, storage_rows = [], []
rng = random.Random(42)

def nvfp4_end_to_end(a, w):
    # 입력은 루프 밖에서 유한성을 검사했다.
    return nvfp4_mm(quantize_nvfp4(a), quantize_nvfp4(w))

for n in SIZES:
    a, w = make_inputs(n, n, n, seed=n)
    aq, wq = prepare(a, w)
    # 각 성능 shape의 일부 출력 행을 FP32로 검증한다. 전체 크기의 참조 저장을 피한다.
    actual = nvfp4_mm(aq, wq)
    a_hat = dequantize_nvfp4(aq)
    w_hat = dequantize_nvfp4(wq)
    sample_rows = torch.linspace(0, n - 1, min(n, 16), device="cuda").long()
    reference = a_hat[sample_rows] @ w_hat.t()
    torch.testing.assert_close(actual[sample_rows].float(), reference, atol=2e-2, rtol=2e-2)
    del actual, a_hat, w_hat, reference
    paths = {"BF16 GEMM": lambda: a @ w.t(), "NVFP4 GEMM": lambda: nvfp4_mm(aq, wq)}
    if n <= E2E_MAX:
        paths.update({"Quantize A+W": lambda: (quantize_nvfp4(a), quantize_nvfp4(w)),
                      "NVFP4 end-to-end": lambda: nvfp4_end_to_end(a, w)})
    for fn in paths.values():
        fn()
    torch.cuda.synchronize()
    samples = {name: [] for name in paths}
    for repeat in range(REPEATS):
        names = list(paths)
        rng.shuffle(names)
        for name in names:
            ms = triton.testing.do_bench(paths[name], warmup=WARMUP_MS,
                                         rep=REP_MS, return_mode="median")
            assert math.isfinite(ms) and ms > 0
            samples[name].append(ms)
    baseline = statistics.median(samples["BF16 GEMM"])
    for name, times in samples.items():
        ms = statistics.median(times)
        benchmark_rows.append(dict(size=n, path=name, ms=ms, min_ms=min(times), max_ms=max(times),
                                   sample_medians_ms=times, speedup_vs_bf16=baseline / ms,
                                   effective_tflops=None if name == "Quantize A+W" else 2 * n**3 / (ms * 1e9)))
    packed_bytes = sum(q.packed.nbytes + q.blocked_scales.nbytes + q.global_scale.nbytes for q in (aq, wq))
    storage_rows.append(dict(size=n, bf16_input_bytes=a.nbytes + w.nbytes,
                             nvfp4_input_bytes=packed_bytes,
                             input_compression=(a.nbytes + w.nbytes) / packed_bytes))
    print(f"{n}: BF16 {baseline:.4f} ms | NVFP4 {statistics.median(samples['NVFP4 GEMM']):.4f} ms", flush=True)
    # 위 저장량은 배포에 필요한 packed 입력만 센다. 논리 scales 사본·임시 버퍼는 제외한다.
    del paths, a, w, aq, wq

bench_df = pd.DataFrame(benchmark_rows)
display(bench_df.drop(columns="sample_medians_ms").round(4))
display(pd.DataFrame(storage_rows))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, frame in bench_df.groupby("path", sort=False):
    frame = frame.sort_values("size")
    axes[0].plot(frame["size"], frame.ms, "o-", label=name)
    if name != "Quantize A+W":
        axes[1].plot(frame["size"], frame.speedup_vs_bf16, "o-", label=name)
for ax in axes:
    ax.set_xscale("log", base=2)
    ax.set_xticks(SIZES, [str(n) for n in SIZES], rotation=45)
    ax.set_xlabel("M = N = K")
    ax.grid(alpha=.25)
    ax.legend(fontsize=8)
axes[0].set_yscale("log")
axes[0].set_ylabel("API latency (ms)")
axes[1].axhline(1, color="gray", linestyle="--")
axes[1].set_ylabel("Speedup over BF16 (higher is better)")
fig.tight_layout()
plt.show()

## 6. 결과를 읽는 법과 실제 커널 확인

작은 행렬에서는 호출 지연과 부족한 병렬성이 큰 비중을 차지할 수 있다. 큰 행렬에서는 낮은 입력 비트 수의 이득과 스케일 로드·타일·연산 파이프라인의 효율이 함께 작용한다. FP4가 BF16보다 항상 일정 배수 빠를 것으로 가정하지 않는다. **속도와 4절의 오차를 함께** 비교한다.

네이티브 FP4 API 호출이 성공했다는 사실과 실제 SASS 명령을 확인했다는 사실은 다르다. 아래 선택 셀은 GPU 프로파일러로 실제 커널 이름과 trace를 저장한다. 명령·occupancy·대역폭의 원인을 더 분석하려면 Nsight Compute에서 같은 경로를 측정한다. G4 sm_120에 sm_100의 `tcgen05` 구현을 그대로 가정하지 않는다.

In [ ]:
RUN_PROFILE = False
kernel_names = []
if RUN_PROFILE:
    from torch.profiler import profile, ProfilerActivity
    a, w = make_inputs(4096, 4096, 4096)
    aq, wq = prepare(a, w)
    nvfp4_mm(aq, wq)
    torch.cuda.synchronize()
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
        nvfp4_mm(aq, wq)
        torch.cuda.synchronize()
    kernel_names = sorted({event.name for event in prof.events()
                           if event.device_type == torch.autograd.DeviceType.CUDA})
    print("\n".join(kernel_names))
    prof.export_chrome_trace("nvfp4_g4_trace.json")
    del a, w, aq, wq
else:
    print("Profiler skipped; set RUN_PROFILE=True to capture actual kernels.")

## 7. 저장

실행 환경, 정확성, 독립 반복의 median, 저장량, 선택적으로 수집한 커널명을 JSON으로 저장한다. Colab 파일 패널에서 내려받을 수 있다. `effective_tflops`는 dense `2MNK` 기준이며 sparse peak와 비교하지 않는다.

참고 구현: [PyTorch FP4 패킹·스케일 배치 테스트 유틸리티](https://github.com/pytorch/pytorch/blob/v2.11.0/torch/testing/_internal/common_quantized.py). 노트북은 내부 테스트 모듈이나 torchao 설치에 의존하지 않도록 필요한 참조 함수를 직접 포함한다.

In [ ]:
assert len(accuracy_rows) == len(CHECK_CASES) and all(row["ok"] for row in accuracy_rows)
assert len(benchmark_rows) == 2 * len(SIZES) + 2 * sum(n <= E2E_MAX for n in SIZES)
report = dict(environment=ENVIRONMENT, started_utc=STARTED_UTC,
              completed_utc=datetime.now(timezone.utc).isoformat(),
              settings=dict(format="NVFP4 E2M1 + E4M3 block16 + FP32 tensor scale",
                            output_dtype="bfloat16", use_fast_accum=False,
                            bf16_reduced_precision_reduction=False, fp32_tf32=False,
                            layout="A row-major; W row-major then transpose view",
                            quantizer="eager reference, nearest ties-to-even",
                            sizes=SIZES, e2e_max=E2E_MAX, repeats=REPEATS,
                            benchmark="do_bench cache-flush median; API output allocation included",
                            warmup_ms=WARMUP_MS, rep_ms=REP_MS),
              format_checks=format_checks, accuracy=accuracy_rows,
              benchmarks=benchmark_rows, storage=storage_rows,
              profiled=RUN_PROFILE, kernel_names=kernel_names)
path = Path("nvfp4_gemm_g4_results.json")
path.write_text(json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False) + "\n")
print("NVFP4_G4_RUN_COMPLETE", path.resolve())

## 8. G4 실측 결과 — 2026-10-03 KST

[전체 측정 JSON](nvfp4_gemm_g4_results.json) · [출력을 보존한 실행 노트북](nvfp4_gemm_g4_output.ipynb)

**RTX PRO 6000 Blackwell Server Edition (sm_120, 188 SM, 94.97 GiB)**에서 실행했다. PyTorch 2.11.0+cu130, CUDA 13.0, Triton 3.6.0이며, 각 경로의 cache-flush median을 3회 측정한 뒤 그 median을 아래에 표시한다. 이 절은 저장된 실행의 고정 스냅샷이며 재실행 시 자동 갱신되지 않는다.

| M=N=K | BF16 GEMM (ms) | NVFP4 GEMM (ms) | 미리 양자화한 NVFP4 속도 향상 | NVFP4 양자화 포함 (ms) |
|---:|---:|---:|---:|---:|
| 128 | 0.008288 | 0.014336 | 0.58× | 0.498688 |
| 256 | 0.010144 | 0.014336 | 0.71× | 0.506464 |
| 512 | 0.014208 | 0.016384 | 0.87× | 0.511840 |
| 1024 | 0.024576 | 0.020480 | 1.20× | 0.531392 |
| 2048 | 0.075744 | 0.040928 | 1.85× | 0.688800 |
| 4096 | 0.370656 | 0.129024 | 2.87× | 미측정 |
| 8192 | 2.728960 | 0.772048 | 3.53× | 미측정 |

작은 128~512에서는 NVFP4 API 호출이 더 느렸고, 1024부터 이 측정의 BF16 기준보다 빨라졌다. 큰 4096/8192에서는 각각 2.87배/3.53배다. 8192의 NVFP4 실행 간 median 범위는 0.770176~0.774656 ms, BF16은 2.715168~2.733088 ms였다. 이 속도 향상은 **이미 패킹·스케일 배치를 마친 입력**의 결과다. 현재 느린 eager 참조 양자화기를 포함하면 2048에서 0.688800 ms로, BF16 0.075744 ms보다 약 **9.09배 느리다**. 4096/8192의 양자화 포함 경로는 기본 `E2E_MAX=2048` 설정에 따라 측정하지 않았다.

CPU·CUDA 형식 검사가 모두 통과했고, 5개 전체 정확성 사례와 7개 벤치마크 shape의 표본 행 비교도 통과했다. 일반 무작위 입력 3개 사례에서 복원된 FP4 입력의 FP32 GEMM 대비 구현 NRMSE는 약 **0.166%**였다. 양자화 전 입력의 FP32 GEMM 대비 최종 FP4 출력의 NRMSE는 **13.39~13.46%**, SQNR은 **17.42~17.46 dB**였다. 따라서 성능 향상에는 입력 정밀도 손실이 수반된다. 원본 BF16 대비 이 오차를 패킹이나 스케일 구현 오류로 해석하지 않는다.

입력 패킹과 스케일의 저장량은 큰 행렬에서 BF16의 약 1/3.56이다. 이 실험만으로 속도 향상의 주원인이 메모리인지 Tensor Core 처리량인지 분리할 수는 없다. 이번 실행은 `RUN_PROFILE=False`였으므로 커널별 SASS·occupancy·메모리 카운터는 수집하지 않았다. A/W layout을 맞춰 같은 노트북에서 측정한 BF16 기준과 비교했으며, 앞 BF16 v6 노트북의 값을 기준으로 나누지 않았다.
